In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

RMSNorm   ok | out RMS 1.000000 | params 768
RMSNorm   fp16 overflow test ok
SwiGLU    ok | hidden 2048 | params 4,718,592


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

attention  shape/finite ok
attention  causality ok
rope       norm preservation ok
rope       relative position ok  (off2 -0.3179=-0.3179, off9 -4.3299)
rope       pos0 offset ok
rope-attn  ok | params 2,359,296


In [3]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

      0.0 MB  __huggingface_repos__.json
      2.3 MB  bpe32k.json
      0.0 MB  bpe32k_meta.json
    434.4 MB  bpe32k_train.bin
      5.1 MB  bpe32k_val.bin
      0.0 MB  corpus_stats.json
      3.0 MB  test.json
    181.6 MB  train_acts.jsonl
    367.1 MB  train_high_courts.jsonl
    296.2 MB  train_supreme_court.jsonl
    183.4 MB  train_tribunals.jsonl
      3.1 MB  val_acts.jsonl
      3.1 MB  val_high_courts.jsonl
      0.9 MB  val_prompts.json
      0.2 MB  val_refs.json
      3.0 MB  val_supreme_court.jsonl
      3.1 MB  val_tribunals.jsonl

vocab 32,000 | pad 0 | eos 3
train 217,201,090 tokens | val 2,560,200 tokens
test 1000 | val prompts 300

sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu


In [11]:
class GeneralizedAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, causal=True, cross=False):
        super().__init__()
        assert dim % n_heads == 0

        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.causal, self.cross = causal, cross

        # separate projections: cross-attn draws q from x, k/v from context
        self.q_proj   = nn.Linear(dim, dim, bias=False)
        self.k_proj   = nn.Linear(dim, dim, bias=False)
        self.v_proj   = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

        # RoPE applies to self-attention only; encoder output is not a
        # sequence the decoder indexes by relative position
        self.rope = None if cross else RoPE(self.head_dim, max_seq)

    def forward(self, x, context=None):
        B, T, C = x.shape
        src = context if self.cross else x
        if self.cross:
            assert context is not None, "cross-attention needs context"
        S = src.shape[1]

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)

        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=(self.causal and not self.cross))
        return self.out_proj(out.transpose(1, 2).reshape(B, T, C))

In [12]:
# ============================================================
# 2. Encoder block
# ============================================================

class EncoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=False,
            cross=False,
        )

        self.norm2 = RMSNorm(dim)

        self.mlp = SwiGLU(dim)

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.mlp(
            self.norm2(x)
        )

        return x


# ============================================================
# 3. Decoder block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()

        # Causal self-attention
        self.norm1 = RMSNorm(dim)

        self.self_attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=True,
            cross=False,
        )

        # Cross-attention to encoder
        self.norm2 = RMSNorm(dim)

        self.cross_attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=False,
            cross=True,
        )

        # MLP
        self.norm3 = RMSNorm(dim)

        self.mlp = SwiGLU(dim)

    def forward(self, x, encoder_output):
        x = x + self.self_attn(
            self.norm1(x)
        )

        x = x + self.cross_attn(
            self.norm2(x),
            context=encoder_output
        )

        x = x + self.mlp(
            self.norm3(x)
        )

        return x

In [9]:
torch.manual_seed(0)
D, H = 768, 12
enc_x, dec_x = torch.randn(2, 64, D), torch.randn(2, 32, D)

eb, db = EncoderBlock(D, H, 2048), DecoderBlock(D, H, 2048)
e = eb(enc_x)
d = db(dec_x, e)
assert e.shape == enc_x.shape and d.shape == dec_x.shape
print(f"enc block {sum(p.numel() for p in eb.parameters()):,} | "
      f"dec block {sum(p.numel() for p in db.parameters()):,}")

# encoder must be bidirectional: changing a LATE token must affect EARLY ones
a = enc_x.clone(); b = enc_x.clone(); b[:, 60] = torch.randn(2, D)
with torch.no_grad():
    assert not torch.allclose(eb(a)[:, 5], eb(b)[:, 5], atol=1e-5), \
        "encoder is not bidirectional"
print("encoder bidirectional ok")

# decoder self-attn must stay causal
a = dec_x.clone(); b = dec_x.clone(); b[:, 20] = torch.randn(2, D)
with torch.no_grad():
    assert torch.allclose(db(a, e)[:, 5], db(b, e)[:, 5], atol=1e-5), \
        "decoder leaks future"
print("decoder causal ok")

enc block 7,079,424 | dec block 9,439,488
encoder bidirectional ok
decoder causal ok


In [22]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIM, N_HEADS, MAX_SEQ = 768, 12, 2048
BLOCK = 1024
print(f"device {DEVICE} | vocab {VOCAB_SIZE:,} | dim {DIM} | heads {N_HEADS}")

device cuda | vocab 32,000 | dim 768 | heads 12


In [20]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

USE_AMP = DEVICE == "cuda"

B_SPLIT, B_ENC_LAYERS, B_DEC_LAYERS = 768, 6, 6


class EncoderDecoderModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_encoder_layers=6, n_decoder_layers=6,
                 n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim, self.max_seq = vocab_size, dim, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)

        self.encoder_blocks = nn.ModuleList(
            [EncoderBlock(dim, n_heads, max_seq) for _ in range(n_encoder_layers)])
        self.norm_e = RMSNorm(dim)

        self.decoder_blocks = nn.ModuleList(
            [DecoderBlock(dim, n_heads, max_seq) for _ in range(n_decoder_layers)])
        self.norm_f = RMSNorm(dim)

        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        std_e = 0.02 / math.sqrt(2 * n_encoder_layers)
        std_d = 0.02 / math.sqrt(2 * n_decoder_layers)

        for b in self.encoder_blocks:
            nn.init.normal_(b.attn.out_proj.weight, 0.0, std_e)
            nn.init.normal_(b.mlp.w3.weight,        0.0, std_e)

        for b in self.decoder_blocks:
            nn.init.normal_(b.self_attn.out_proj.weight,  0.0, std_d)
            nn.init.normal_(b.cross_attn.out_proj.weight, 0.0, std_d)
            nn.init.normal_(b.mlp.w3.weight,              0.0, std_d)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, split=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")
        split = split or T // 2
        if not 1 <= split <= T:
            raise ValueError(f"invalid split={split} for T={T}")

        e = self.tok_emb(idx[:, :split])
        for b in self.encoder_blocks:
            e = b(e)
        e = self.norm_e(e)                      # normalise once, before cross-attn

        d = self.tok_emb(idx[:, split - 1:])    # start one early -> first pred is at `split`
        for b in self.decoder_blocks:
            d = b(d, e)
        logits = self.head(self.norm_f(d))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets[:, split - 1:].reshape(-1))
        return logits, loss


def count_unique_params(m):
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())


# ============================================================
# CHECKS
# ============================================================

torch.manual_seed(42)
model_b = EncoderDecoderModel(
    vocab_size=VOCAB_SIZE, dim=DIM,
    n_encoder_layers=B_ENC_LAYERS, n_decoder_layers=B_DEC_LAYERS,
    n_heads=N_HEADS, max_seq=MAX_SEQ).to(DEVICE)

# ---- 1. parameter count ----
b_params = count_unique_params(model_b)
emb = model_b.tok_emb.weight.numel()
enc = sum(count_unique_params(b) for b in model_b.encoder_blocks)
dec = sum(count_unique_params(b) for b in model_b.decoder_blocks)

print(f"params {b_params:,} ({b_params/1e6:.2f}M)")
assert 50_000_000 <= b_params <= 300_000_000, f"{b_params:,} outside 50M-300M"
print(f"  embedding (tied) {emb:,}")
print(f"  encoder x{B_ENC_LAYERS}     {enc:,}")
print(f"  decoder x{B_DEC_LAYERS}     {dec:,}")

# ---- 2. initial loss ----
torch.manual_seed(123)
idx_t = torch.randint(0, VOCAB_SIZE, (4, 1024), device=DEVICE)
tgt_t = torch.randint(0, VOCAB_SIZE, (4, 1024), device=DEVICE)

model_b.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    _, loss_b = model_b(idx_t, tgt_t, split=B_SPLIT)
loss_b = loss_b.item()

print(f"\ninitial loss {loss_b:.4f} vs ln(vocab) {math.log(VOCAB_SIZE):.4f}")
assert 9.0 < loss_b < 12.0, f"unexpected init loss {loss_b:.4f}"

# ---- 3. overfit one real batch ----
model_b.train()
rng = np.random.RandomState(12345)
off = rng.randint(0, len(train_bin) - 1025, size=4)

x_of = torch.from_numpy(
    np.stack([train_bin[i:i + 1024] for i in off]).astype(np.int64)).to(DEVICE)
y_of = torch.from_numpy(
    np.stack([train_bin[i + 1:i + 1 + 1024] for i in off]).astype(np.int64)).to(DEVICE)

opt = torch.optim.AdamW(model_b.parameters(), lr=1e-3, weight_decay=0.0)
sc  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
hist = []

for step in range(100):
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        _, loss = model_b(x_of, y_of, split=B_SPLIT)
    sc.scale(loss).backward()
    sc.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
    sc.step(opt); sc.update()
    hist.append(loss.item())
    if step == 0 or (step + 1) % 20 == 0:
        print(f"  overfit {step+1:3d}/100  loss {loss.item():.4f}")

print(f"\noverfit {hist[0]:.4f} -> {hist[-1]:.4f}")
assert hist[-1] < 1.0, f"gradients not flowing: {hist[-1]:.4f}"

# ---- 4. encoder-matters test, with a decoder-side control ----
model_b.eval()
torch.manual_seed(7)
base = torch.randint(0, VOCAB_SIZE, (1, 1024), device=DEVICE)


def delta(pos):
    pert = base.clone()
    pert[0, pos] = (base[0, pos].item() + 1) % VOCAB_SIZE
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        a, _ = model_b(base, split=B_SPLIT)
        b, _ = model_b(pert, split=B_SPLIT)
    return (a.float() - b.float()).abs().max().item()


enc_d, dec_d = delta(100), delta(900)
print(f"\nsingle-token perturbation: encoder pos 100 -> {enc_d:.6f} | "
      f"decoder pos 900 -> {dec_d:.6f}")

# single-token encoder influence sits at the fp16 floor once the model has
# memorised the batch, so test the whole prompt instead
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    a, _ = model_b(base, split=B_SPLIT)
    alt = base.clone()
    alt[0, :B_SPLIT] = torch.randint(0, VOCAB_SIZE, (B_SPLIT,), device=DEVICE)
    b, _ = model_b(alt, split=B_SPLIT)

full_swap = (a.float() - b.float()).abs().max().item()
print(f"full encoder swap -> {full_swap:.4f}")
assert full_swap > 0.5, "encoder genuinely dead"
print("encoder influence ok")

del opt, sc, x_of, y_of
torch.cuda.empty_cache()

print(f"\nMODEL B READY — {b_params/1e6:.2f}M params "
      f"({100*(b_params-109_529_856)/109_529_856:+.1f}% vs A)")
print(f"split {B_SPLIT}: encoder reads {B_SPLIT}, decoder predicts {1024-B_SPLIT} "
      f"tokens per window")

params 123,691,008 (123.69M)
  embedding (tied) 24,576,000
  encoder x6     42,476,544
  decoder x6     56,636,928

initial loss 10.5488 vs ln(vocab) 10.3735
  overfit   1/100  loss 10.4763
  overfit  20/100  loss 4.8994
  overfit  40/100  loss 1.2854
  overfit  60/100  loss 0.0083
  overfit  80/100  loss 0.0013
  overfit 100/100  loss 0.0007

overfit 10.4763 -> 0.0007

single-token perturbation: encoder pos 100 -> 0.008301 | decoder pos 900 -> 5.185547
full encoder swap -> 5.3359
encoder influence ok

MODEL B READY — 123.69M params (+12.9% vs A)
split 768: encoder reads 768, decoder predicts 256 tokens per window


In [ ]:
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    a, _ = model_b(base, split=B_SPLIT)

    # swap the entire encoder half for different tokens
    alt = base.clone()
    alt[0, :B_SPLIT] = torch.randint(0, VOCAB_SIZE, (B_SPLIT,), device=DEVICE)
    b, _ = model_b(alt, split=B_SPLIT)

full_swap = (a.float() - b.float()).abs().max().item()
print(f"full encoder swap -> max logit delta {full_swap:.4f}")
assert full_swap > 0.5, "encoder genuinely dead"
print("encoder influence ok")

In [ ]:
print(f"enc_d {enc_d:.6f}  dec_d {dec_d:.6f}  ratio {enc_d/dec_d:.4f}")

In [21]:
import os, time, math, random
import numpy as np
import torch

BLOCK, MICRO_BS, ACCUM = 1024, 8, 4
TOKENS_PER_STEP = MICRO_BS * ACCUM * BLOCK

LR, MIN_LR, WARMUP, WD = 6e-4, 6e-5, 200, 0.1

MAX_STEPS = 6600                      # identical to A — matched token budget
EVAL_EVERY, SAVE_EVERY, LOG_EVERY = 250, 250, 50
VAL_BATCHES, SEED = 20, 42

CKPT_PATH    = "/kaggle/working/ckpt/B_encdec_last.pt"
WEIGHTS_PATH = "/kaggle/working/ckpt/B_encdec.pt"
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

USE_AMP = DEVICE == "cuda"

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# rebuild fresh — the checks above left model_b memorised on 4 sequences
model_b = EncoderDecoderModel(
    vocab_size=VOCAB_SIZE, dim=DIM,
    n_encoder_layers=B_ENC_LAYERS, n_decoder_layers=B_DEC_LAYERS,
    n_heads=N_HEADS, max_seq=MAX_SEQ).to(DEVICE)

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

batch_rng = np.random.RandomState(SEED + 1)


def get_batch(data, rng, bs):
    ix = rng.randint(0, len(data) - BLOCK - 1, size=bs)
    x = np.stack([data[i:i + BLOCK]         for i in ix]).astype(np.int64)
    y = np.stack([data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
    return (torch.from_numpy(x).to(DEVICE, non_blocking=True),
            torch.from_numpy(y).to(DEVICE, non_blocking=True))


def make_val_batches():
    """Same seed as A -> identical validation windows, comparable losses."""
    rng, out = np.random.RandomState(SEED + 2), []
    for _ in range(VAL_BATCHES):
        ix = rng.randint(0, len(val_data) - BLOCK - 1, size=MICRO_BS)
        x = np.stack([val_data[i:i + BLOCK]         for i in ix]).astype(np.int64)
        y = np.stack([val_data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
        out.append((torch.from_numpy(x), torch.from_numpy(y)))
    return out


val_batches = make_val_batches()

decay   = [p for p in model_b.parameters() if p.requires_grad and p.dim() >= 2]
nodecay = [p for p in model_b.parameters() if p.requires_grad and p.dim() < 2]

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WD},
     {"params": nodecay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95), fused=(DEVICE == "cuda"))

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def get_lr(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    p = min(max((step - WARMUP) / (MAX_STEPS - WARMUP), 0.0), 1.0)
    return MIN_LR + (LR - MIN_LR) * 0.5 * (1.0 + math.cos(math.pi * p))


def save_checkpoint(step, hist):
    state = dict(model=model_b.state_dict(), optimizer=optimizer.state_dict(),
                 scaler=scaler.state_dict(), step=step, loss_history=hist,
                 batch_rng_state=batch_rng.get_state(),
                 torch_rng_state=torch.get_rng_state(),
                 cuda_rng_state=(torch.cuda.get_rng_state_all()
                                 if torch.cuda.is_available() else None),
                 config=dict(arch="encoder-decoder", split=B_SPLIT,
                             n_enc=B_ENC_LAYERS, n_dec=B_DEC_LAYERS,
                             block=BLOCK, micro_bs=MICRO_BS, accum=ACCUM,
                             lr=LR, min_lr=MIN_LR, warmup=WARMUP,
                             wd=WD, max_steps=MAX_STEPS))
    torch.save(state, CKPT_PATH + ".tmp")
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)

    fp16 = {k: v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()
            for k, v in model_b.state_dict().items()}
    torch.save(fp16, WEIGHTS_PATH + ".tmp")
    os.replace(WEIGHTS_PATH + ".tmp", WEIGHTS_PATH)


start_step, loss_history = 0, []

if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model_b.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_step   = ck["step"] + 1
    loss_history = ck.get("loss_history", [])
    batch_rng.set_state(ck["batch_rng_state"])
    torch.set_rng_state(ck["torch_rng_state"].cpu()
                        if torch.is_tensor(ck["torch_rng_state"]) else ck["torch_rng_state"])
    if torch.cuda.is_available() and ck.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(ck["cuda_rng_state"])
    print(f"resumed from step {start_step}", flush=True)
else:
    print("starting fresh", flush=True)


@torch.no_grad()
def evaluate():
    model_b.eval()
    total = 0.0
    for xc, yc in val_batches:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, l = model_b(xc.to(DEVICE), yc.to(DEVICE), split=B_SPLIT)
        total += l.item()
    model_b.train()
    return total / len(val_batches)


model_b.train()
print(f"device {DEVICE} | tokens/step {TOKENS_PER_STEP:,} | max_steps {MAX_STEPS} | "
      f"split {B_SPLIT} (loss on {BLOCK - B_SPLIT} positions/window)\n", flush=True)

warmup_done_t, warmup_done_step = None, start_step + 3
t_run = time.time()

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum = torch.zeros((), device=DEVICE)

    for _ in range(ACCUM):
        x, y = get_batch(train_data, batch_rng, MICRO_BS)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, loss = model_b(x, y, split=B_SPLIT)
        scaler.scale(loss / ACCUM).backward()
        accum += loss.detach()

    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
    scaler.step(optimizer); scaler.update()

    step_loss = (accum / ACCUM).item()
    loss_history.append(step_loss)

    if step == warmup_done_step:
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        warmup_done_t = time.time()

    if (step + 1) % LOG_EVERY == 0:
        if warmup_done_t and step > warmup_done_step:
            tps = ((step - warmup_done_step) * TOKENS_PER_STEP
                   / (time.time() - warmup_done_t))
            eta = (MAX_STEPS - step - 1) * TOKENS_PER_STEP / tps / 3600
            tps_s, eta_s = f"{tps:,.0f}", f"{eta:4.2f}h"
        else:
            tps_s, eta_s = "warmup", "  -  "
        print(f"step {step+1:6d}/{MAX_STEPS} | loss {step_loss:7.4f} | lr {lr:.2e} | "
              f"grad {float(gnorm):6.3f} | tok/s {tps_s} | "
              f"{(time.time()-t_run)/60:6.1f} min | eta {eta_s}", flush=True)

    if (step + 1) % EVAL_EVERY == 0:
        vl = evaluate()
        print(f"  >> val loss {vl:.4f} (ppl {math.exp(vl):.1f})", flush=True)

    if (step + 1) % SAVE_EVERY == 0 or step + 1 == MAX_STEPS:
        save_checkpoint(step, loss_history)
        print(f"  >> checkpoint @ step {step+1}", flush=True)

print(f"\nTRAINING COMPLETE — {step+1} steps, "
      f"{(step+1)*TOKENS_PER_STEP/1e6:.0f}M tokens, "
      f"{(time.time()-t_run)/3600:.2f} h", flush=True)
print(f"final train loss {step_loss:.4f} | final val loss {evaluate():.4f}", flush=True)

starting fresh
device cuda | tokens/step 32,768 | max_steps 6600 | split 768 (loss on 256 positions/window)

step     50/6600 | loss  6.8965 | lr 1.50e-04 | grad  1.583 | tok/s 19,487 |    1.4 min | eta 3.06h
step    100/6600 | loss  5.9662 | lr 3.00e-04 | grad  0.991 | tok/s 20,170 |    2.7 min | eta 2.93h
step    150/6600 | loss  5.2069 | lr 4.50e-04 | grad  1.042 | tok/s 20,499 |    4.0 min | eta 2.86h


KeyboardInterrupt: 